In [0]:
adls_options = {
    "fs.azure.account.auth.type.adlsprimework2.dfs.core.windows.net": "OAuth",
    "fs.azure.account.oauth.provider.type.adlsprimework2.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id.adlsprimework2.dfs.core.windows.net": "<CLIENT_ID>",
    "fs.azure.account.oauth2.client.secret.adlsprimework2.dfs.core.windows.net": "<CLIENT_SECRET>",
    "fs.azure.account.oauth2.client.endpoint.adlsprimework2.dfs.core.windows.net": "<TENANT_ENDPOINT>"
}

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import * 

In [0]:
df_gold = (
    spark.read.format("delta")
    .options(**adls_options)
    .option("header","true")
    .option("inferSchema","true")
    .load(
        "abfss://silver@adlsprimework2.dfs.core.windows.net/amazon_prime_titles_silver.csv"
    )
)

display(df_gold)

show_id,type,content_title,director,cast,country,date_added,release_year,rating,duration,listed_in
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,June 20 2021,2001,ALL,1 Season,Kids
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,June 22 2021,2020,7+,1 Season,Anime
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,June 3 2021,2019,16+,96 min,Comedy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,June 25 2021,1941,NR,62 min,Comedy
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,April 4 2021,2016,ALL,88 min,Adventure Kids
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,June 30 2021,1991,R,94 min,Drama
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import DateType

@udf(DateType())
def parse_date_str(s):
    from datetime import datetime
    if not s:
        return None
    _M = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
          "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
    try:
        parts = s.strip().split()
        return datetime(int(parts[2]), _M[parts[0]], int(parts[1])).date()
    except:
        return None

df_gold = df_gold.withColumn(
    "date_added",
    parse_date_str(col("date_added"))
)

df_gold = df_gold.withColumn(
    "year_added",
    year(col("date_added"))
)

display(df_gold)

show_id,type,content_title,director,cast,country,date_added,release_year,rating,duration,listed_in,year_added
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,2021-06-20,2001,ALL,1 Season,Kids,2021
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,2021-03-30,1989,ALL,45 min,Fantasy Drama,2021
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,2021-06-22,2020,7+,1 Season,Anime,2021
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,2021-06-03,2019,16+,96 min,Comedy Drama,2021
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,2021-03-30,1989,ALL,52 min,Kids Fantasy,2021
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,2021-06-25,1941,NR,62 min,Comedy,2021
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,2021-04-04,2016,ALL,88 min,Adventure Kids,2021
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,2021-03-30,2017,16+,98 min,Documentary,2021
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,2021-06-30,1991,R,94 min,Drama,2021
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,2021-03-30,2016,13+,131 min,Comedy,2021


In [0]:
df_gold = df_gold.withColumn(
    "category_1",
    split(col("listed_in"), ",")[0]
)

df_gold = df_gold.withColumn(
    "category_2",
    when(
        get(split(col("listed_in"), ","), 1).isNull(),
        "Unknown"
    ).otherwise(get(split(col("listed_in"), ","), 1))
)

display(df_gold)

show_id,type,content_title,director,cast,country,date_added,release_year,rating,duration,listed_in,year_added,category_1,category_2
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,2021-06-20,2001,ALL,1 Season,Kids,2021,Kids,Unknown
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,2021-03-30,1989,ALL,45 min,Fantasy Drama,2021,Fantasy Drama,Unknown
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,2021-06-22,2020,7+,1 Season,Anime,2021,Anime,Unknown
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,2021-06-03,2019,16+,96 min,Comedy Drama,2021,Comedy Drama,Unknown
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,2021-03-30,1989,ALL,52 min,Kids Fantasy,2021,Kids Fantasy,Unknown
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,2021-06-25,1941,NR,62 min,Comedy,2021,Comedy,Unknown
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,2021-04-04,2016,ALL,88 min,Adventure Kids,2021,Adventure Kids,Unknown
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,2021-03-30,2017,16+,98 min,Documentary,2021,Documentary,Unknown
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,2021-06-30,1991,R,94 min,Drama,2021,Drama,Unknown
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,2021-03-30,2016,13+,131 min,Comedy,2021,Comedy,Unknown


In [0]:
df_gold = df_gold.withColumn(
    "category_2",
    when(
        col("category_2").isNull(),
        "Unknown"
    ).otherwise(df_gold["category_2"])
)

display(df_gold)

show_id,type,content_title,director,cast,country,date_added,release_year,rating,duration,listed_in,year_added,category_1,category_2
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,2021-06-20,2001,ALL,1 Season,Kids,2021,Kids,Unknown
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,2021-03-30,1989,ALL,45 min,Fantasy Drama,2021,Fantasy Drama,Unknown
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,2021-06-22,2020,7+,1 Season,Anime,2021,Anime,Unknown
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,2021-06-03,2019,16+,96 min,Comedy Drama,2021,Comedy Drama,Unknown
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,2021-03-30,1989,ALL,52 min,Kids Fantasy,2021,Kids Fantasy,Unknown
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,2021-06-25,1941,NR,62 min,Comedy,2021,Comedy,Unknown
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,2021-04-04,2016,ALL,88 min,Adventure Kids,2021,Adventure Kids,Unknown
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,2021-03-30,2017,16+,98 min,Documentary,2021,Documentary,Unknown
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,2021-06-30,1991,R,94 min,Drama,2021,Drama,Unknown
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,2021-03-30,2016,13+,131 min,Comedy,2021,Comedy,Unknown


In [0]:
df_gold.write.format("delta") \
       .options(**adls_options) \
       .mode("append") \
       .option(
           "path",
           "abfss://gold@adlsprimework2.dfs.core.windows.net/amazon_prime_titles_gold.csv"
       ) \
       .save()

In [0]:
%sql
create database if not exists gold_layer;

In [0]:
%sql
DROP SCHEMA IF EXISTS gold_layer;

In [0]:
df_gold.write.format("delta")\
    .mode("append")\
    .saveAsTable('gold_layer.prime_gold')

In [0]:
spark.sql("""

select * from gold_layer.prime_gold""")

DataFrame[show_id: string, type: string, content_title: string, director: string, cast: string, country: string, date_added: date, release_year: int, rating: string, duration: string, listed_in: string, year_added: int, category_1: string, category_2: string]